In [ ]:
# Libraries used for proprecessing
import numpy as np
import pandas as pd
import math
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from einops import rearrange

# Libraries used to plot graphs
import matplotlib.pyplot as plt

# Libraries used to display progress bars
from tqdm import tqdm

# Libraries used to import datasets from Kaggle
import os

# Libraries used to take care of warnings
import warnings

# Libraries used to create the neural network
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# List of categorical and numerical features
from config import categorical_columns, numerical_columns  # config.py has been intentionally omitted from this repository

# Environnement settings

In [3]:
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data import

In [ ]:
data_path_X_train='Dataset_data_challenge/X_train_preprocessed.csv'
X_train=pd.read_csv(data_path_X_train, sep=',')

data_path_X_test='Dataset_data_challenge/X_test_preprocessed.csv'
X_test=pd.read_csv(data_path_X_test, sep=',')

data_path_y_train='Dataset_data_challenge/y_train.csv'
y_train=pd.read_csv(data_path_y_train, sep=',')

# Reshaping Data for the TFT

In [ ]:
# Creating a 3d array by grouping sequences for train data
X_train_reshaped= np.zeros((int(X_train.shape[0]/100), 100, 11))
for i in tqdm(range(int(X_train.shape[0]/100)), desc="X_train", colour='Green'):
    seq_start = i * 100
    seq_end = (i + 1) * 100
    X_train_reshaped[i] = X_train.iloc[seq_start:seq_end][X_train.columns].to_numpy() 

# Same for test data
X_test_reshaped= np.zeros((int(X_test.shape[0]/100), 100, 11))
for i in tqdm(range(int(X_test.shape[0]/100)), desc="X_test", colour='Red'):
    seq_start = i * 100
    seq_end = (i + 1) * 100
    X_test_reshaped[i] = X_test.iloc[seq_start:seq_end][X_test.columns].to_numpy() 


# Creation of the datasets and dataloaders

In [ ]:
# Train and validation dataset
train_indices, val_indices = train_test_split(np.arange(len(y_train)), test_size=0.1, random_state=42, stratify=y_train)

class TrainDataset(Dataset):
   def __init__(self, X, y, indices):
       self.X = X
       self.y = y
       self.indices = indices

   def __len__(self):
       return len(self.indices)

   def __getitem__(self, idx):
       i = self.indices[idx]
       return torch.tensor(self.X[i], dtype=torch.float32), torch.tensor(self.y[i], dtype=torch.long)

train_dataset = TrainDataset(X_train_reshaped, y_train, train_indices)
val_dataset = TrainDataset(X_train_reshaped, y_train, val_indices)

# Test dataset (for prediction)
class TestDataset(Dataset):
   def __init__(self, X):
       self.X = X

   def __len__(self):
       return len(self.X)

   def __getitem__(self, idx):
       return torch.tensor(self.X[idx], dtype=torch.float32)

test_dataset = TestDataset(X_test_reshaped)

# Creation of Data Loaders
batch_size = 128 #250
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# TFT Utility Modules

In [ ]:
class TimeDistributed(nn.Module):
   def __init__(self, module, batch_first=False):
       super(TimeDistributed, self).__init__()
       self.module = module
       self.batch_first = batch_first

   def forward(self, x):
       if len(x.size()) <= 2:
           return self.module(x)
       # Squash samples and timesteps into a single axis
       x_reshape = x.contiguous().view(-1, x.size(-1))
       y = self.module(x_reshape)
       # We have to reshape Y
       if self.batch_first:
           y = y.contiguous().view(x.size(0), -1, y.size(-1))
       else:
           y = y.view(-1, x.size(1), y.size(-1))
       return y

class GLU(nn.Module):
   def __init__(self, input_size):
       super(GLU, self).__init__()
       self.fc1 = nn.Linear(input_size,input_size)
       self.fc2 = nn.Linear(input_size, input_size)
       self.sigmoid = nn.Sigmoid()

   def forward(self, x):
       sig = self.sigmoid(self.fc1(x))
       x = self.fc2(x)
       return torch.mul(sig, x)

class GatedResidualNetwork(nn.Module):
   def __init__(self, input_size, hidden_state_size, output_size, dropout, hidden_context_size=None, batch_first=False):
       super(GatedResidualNetwork, self).__init__()
       self.input_size = input_size
       self.output_size = output_size
       self.hidden_context_size = hidden_context_size
       self.hidden_state_size=hidden_state_size
       self.dropout_val = dropout

       if self.input_size!=self.output_size:
           self.skip_layer = TimeDistributed(nn.Linear(self.input_size, self.output_size), batch_first=batch_first)
       else:
           self.skip_layer = None

       self.fc1 = TimeDistributed(nn.Linear(self.input_size, self.hidden_state_size), batch_first=batch_first)
       self.elu1 = nn.ELU()

       if self.hidden_context_size is not None:
           self.context = TimeDistributed(nn.Linear(self.hidden_context_size, self.hidden_state_size),batch_first=batch_first)
       else:
           self.context = None

       self.fc2 = TimeDistributed(nn.Linear(self.hidden_state_size,  self.output_size), batch_first=batch_first)
       self.elu2 = nn.ELU()

       self.dropout = nn.Dropout(self.dropout_val)
       self.bn = TimeDistributed(nn.BatchNorm1d(self.output_size),batch_first=batch_first)
       self.gate = TimeDistributed(GLU(self.output_size), batch_first=batch_first)

   def forward(self, x, context=None):
       if self.skip_layer is not None:
           residual = self.skip_layer(x)
       else:
           residual = x

       x = self.fc1(x)
       if context is not None and self.context is not None:
           context = self.context(context)
           x = x+context
       x = self.elu1(x)

       x = self.fc2(x)
       x = self.dropout(x)
       x = self.gate(x)
       x = x+residual
       x = self.bn(x)

       return x

class VariableSelectionNetwork(nn.Module):
   def __init__(self, input_size, num_inputs, hidden_size, dropout, context_size=None):
       super(VariableSelectionNetwork, self).__init__()
       self.hidden_size = hidden_size
       self.input_size =input_size
       self.num_inputs = num_inputs
       self.dropout_val = dropout
       self.context_size = context_size

       # Flattened GRN to get weights
       self.flattened_grn = GatedResidualNetwork(
           self.num_inputs*self.input_size,
           self.hidden_size,
           self.num_inputs,
           self.dropout_val,
           hidden_context_size=self.context_size
       )

       self.single_variable_grns = nn.ModuleList()
       for i in range(self.num_inputs):
           self.single_variable_grns.append(
               GatedResidualNetwork(self.input_size, self.hidden_size, self.hidden_size, self.dropout_val)
           )

       self.softmax = nn.Softmax(dim=-1)

   def forward(self, embedding, context=None):

       if context is not None:
           sparse_weights = self.flattened_grn(embedding, context)
       else:
           sparse_weights = self.flattened_grn(embedding)

       sparse_weights = self.softmax(sparse_weights).unsqueeze(3)  

       var_outputs = []
       for i in range(self.num_inputs):
           start_idx = i*self.input_size
           end_idx = (i+1)*self.input_size
           var_i = embedding[:,:, start_idx:end_idx] 
           var_out = self.single_variable_grns[i](var_i)
           var_outputs.append(var_out)

       var_outputs = torch.stack(var_outputs, axis=2)

       outputs = var_outputs*sparse_weights  
       outputs = outputs.sum(axis=2)

       return outputs, sparse_weights

class PositionalEncoder(nn.Module):
   def __init__(self, d_model, max_seq_len=5000):
       super().__init__()
       self.d_model = d_model
       pe = torch.zeros(max_seq_len, d_model)
       position = torch.arange(0, max_seq_len).unsqueeze(1)
       div_term = torch.exp(torch.arange(0, d_model, 2)* (-math.log(10000.0)/d_model))
       pe[:,0::2] = torch.sin(position * div_term)
       pe[:,1::2] = torch.cos(position * div_term)
       pe = pe.unsqueeze(0) 
       self.register_buffer('pe', pe)

   def forward(self, x):
       seq_len = x.size(1)
       x = x * math.sqrt(self.d_model)
       x = x + self.pe[:,:seq_len,:]
       return x

# TFT model

In [ ]:
class TFTClassifier(nn.Module):
   def __init__(self,
                cat_dims,       
                num_numerical,  
                num_classes=24,
                embedding_size=8,   
                hidden_size=128,
                lstm_layers=1,
                dropout=0.1,
                attn_heads=4,
                max_seq_len=100):
       super().__init__()

       self.cat_dims = cat_dims
       self.num_cats = len(cat_dims)
       self.num_numerical = num_numerical
       self.embedding_size = embedding_size
       self.hidden_size = hidden_size
       self.lstm_layers = lstm_layers
       self.dropout_val = dropout
       self.attn_heads = attn_heads
       self.max_seq_len = max_seq_len
       self.num_classes = num_classes

       # Embeddings of categorical variables
       self.cat_embeddings = nn.ModuleList([
           nn.Embedding(cat_dim, self.embedding_size) for cat_dim in cat_dims
       ])

       # Linear layer for numerical features (we transform each numerical variable into an embedding of size embedding_size)
       self.num_linear = nn.ModuleList([
           nn.Linear(1, self.embedding_size) for _ in range(num_numerical)
       ])

       self.input_dim = self.num_cats*self.embedding_size + self.num_numerical*self.embedding_size

       # Variable Selection Network
       self.vsn = VariableSelectionNetwork(input_size=self.embedding_size,
                                            num_inputs=(self.num_cats+self.num_numerical),
                                            hidden_size=self.hidden_size,
                                            dropout=self.dropout_val)

       # LSTM Encoder
       # self.lstm = nn.LSTM(input_size=self.hidden_size,
       #                     hidden_size=self.hidden_size,
       #                     num_layers=self.lstm_layers,
       #                     dropout=(self.dropout_val if self.lstm_layers>1 else 0),
       #                     batch_first=True)
       self.lstm = nn.LSTM(input_size=self.hidden_size,
                   hidden_size=self.hidden_size,
                   num_layers=self.lstm_layers,
                   dropout=(self.dropout_val if self.lstm_layers>1 else 0),
                   batch_first=True,
                   bidirectional=True)
       self.lstm_reduce = nn.Linear(self.hidden_size*2, self.hidden_size)

       self.post_lstm_gate = TimeDistributed(GLU(self.hidden_size), batch_first=True)
       self.post_lstm_norm = TimeDistributed(nn.BatchNorm1d(self.hidden_size), batch_first=True)

       self.static_enrichment = GatedResidualNetwork(self.hidden_size, self.hidden_size, self.hidden_size, self.dropout_val, batch_first=True)

       # Positional encoding
       self.position_encoding = PositionalEncoder(d_model=self.hidden_size, max_seq_len=self.max_seq_len)

       # Multihead attention
       self.multihead_attn = nn.MultiheadAttention(self.hidden_size, self.attn_heads, dropout=self.dropout_val, batch_first=True)
       self.post_attn_gate = TimeDistributed(GLU(self.hidden_size), batch_first=True)
       self.post_attn_norm = TimeDistributed(nn.BatchNorm1d(self.hidden_size), batch_first=True)

       # Feed Forward
       self.pos_wise_ff = GatedResidualNetwork(self.hidden_size, self.hidden_size, self.hidden_size, self.dropout_val, batch_first=True)

       # Output
       self.pre_output_norm = TimeDistributed(nn.BatchNorm1d(self.hidden_size), batch_first=True)
       self.pre_output_gate = TimeDistributed(GLU(self.hidden_size), batch_first=True)
       self.output_layer = nn.Linear(self.hidden_size, self.num_classes)

   def forward(self, x):

       B,T,F = x.size()

       cat_part = x[:,:,:self.num_cats].long() 
       num_part = x[:,:,self.num_cats:].float() 

       cat_emb_list = []
       for i, emb in enumerate(self.cat_embeddings):
           cat_emb_list.append(emb(cat_part[:,:,i]))
       cat_emb = torch.cat(cat_emb_list, dim=-1) 

       num_emb_list = []
       for i, lin in enumerate(self.num_linear):
           feat = num_part[:,:,i].unsqueeze(-1) 
           num_emb_list.append(lin(feat))
       num_emb = torch.cat(num_emb_list, dim=-1) 

       # Concat cat_emb et num_emb
       full_emb = torch.cat([cat_emb, num_emb], dim=-1) 

       # VSN
       vsn_out, sparse_weights = self.vsn(full_emb)

       # Positionnal encoding
       vsn_out = self.position_encoding(vsn_out) 
       # LSTM
       lstm_out, _ = self.lstm(vsn_out) 
       lstm_out = self.lstm_reduce(lstm_out) 


       # Skip-connection + normalisation
       lstm_out = self.post_lstm_gate(lstm_out + vsn_out)
       lstm_out = self.post_lstm_norm(lstm_out)
       attn_in = self.static_enrichment(lstm_out)

       # Attention
       # multihead_attn batch_first=True => attn_in : (B,T,H)
       attn_out, attn_weights = self.multihead_attn(attn_in, attn_in, attn_in)
       attn_out = self.post_attn_gate(attn_out) + attn_in
       attn_out = self.post_attn_norm(attn_out)

       # Position-wise FF
       ff_out = self.pos_wise_ff(attn_out)
       ff_out = self.pre_output_gate(ff_out) + attn_out
       ff_out = self.pre_output_norm(ff_out)

       # Temporal dimension pooling (average)for classification
       pooled = ff_out.mean(dim=1)

       # Output layer
       logits = self.output_layer(pooled) 
       return logits

# Settings of the model

In [ ]:
cat_dims = [X_train[col].max()+1 for col in categorical_columns]
embedding_size = 8  
num_numerical = len(numerical_columns)
num_classes = 24 

model = TFTClassifier(
   cat_dims=cat_dims,
   num_numerical=num_numerical,
   embedding_size=8,     
   hidden_size=600,       
   lstm_layers=3,         
   dropout=0.4,           
   attn_heads=60,         
   max_seq_len=100,       
   num_classes=num_classes
).to(device)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00035, weight_decay=4.6073260181584066e-05)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=3)

Nombre de paramètres entraînables : 45692332


# Training

In [ ]:
def train_one_epoch(model, train_loader, optimizer, criterion):
   model.train()
   total_loss = 0
   total_correct = 0
   total = 0
   for X, y in tqdm(train_loader, desc="Training", colour='Green'):
       X, y = X.to(device), y.to(device)
       optimizer.zero_grad()
       out = model(X)
       loss = criterion(out, y)
       loss.backward()
       optimizer.step()

       total_loss += loss.item() * X.size(0)
       preds = out.argmax(dim=1)
       total_correct += (preds == y).sum().item()
       total += X.size(0)
   return total_loss/total, total_correct/total

def eval_model(model, val_loader, criterion):
   model.eval()
   total_loss = 0
   total_correct = 0
   total = 0
   with torch.no_grad():
       for X, y in tqdm(val_loader, desc="Validation", colour='Blue'):
           X, y = X.to(device), y.to(device)
           out = model(X)
           loss = criterion(out, y)
           total_loss += loss.item() * X.size(0)
           preds = out.argmax(dim=1)
           total_correct += (preds == y).sum().item()
           total += X.size(0)
   return total_loss/total, total_correct/total

best_val_acc = 0
patience = 5
counter = 0

train_loss_list=[]
train_acc_list=[]
val_loss_list=[]
val_acc_list=[]

for epoch in range(40):
   train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
   val_loss, val_acc = eval_model(model, val_loader, criterion)
   scheduler.step(val_loss)
   train_loss_list.append(train_loss)
   train_acc_list.append(train_acc)
   val_loss_list.append(val_loss)
   val_acc_list.append(val_acc)

   print(f"Epoch {epoch+1}: Train loss {train_loss:.4f}, Train Acc {train_acc:.4f}, Val loss {val_loss:.4f}, Val Acc {val_acc:.4f}")

   if val_acc > best_val_acc:
       best_val_acc = val_acc # Save the best value of accuracy on the validation dataset
       counter = 0
       torch.save(model.state_dict(), "best_model_tft.pth")  # Save the model that gave the best accuracy
   else:
       counter += 1
       if counter > patience: # If the accuracy on the validation dataset did not improve for a certain number of epoch we stop
           print("Early stopping!")
           break

# Graphs of accuracy and loss

In [ ]:
plt.subplot(2, 1, 1)
plt.plot(train_loss_list, label='Train Loss')
plt.plot(val_loss_list, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.subplot(2, 1, 2)
plt.plot(train_acc_list, label='Train Accuracy')
plt.plot(val_acc_list, label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Prediction on test data

In [ ]:
model.load_state_dict(torch.load("best_model_tft.pth"))
model.eval()
all_preds = []
with torch.no_grad():
   for X in tqdm(test_loader, desc="Validation", colour='Red'):
       X = X.to(device)
       out = model(X)
       preds = out.argmax(dim=1).cpu().numpy()
       all_preds.extend(preds)

def transfo_format_predi(y_pred):
    y_pred_soumission = np.column_stack((np.arange(len(y_pred)), y_pred))
    df_y_pred_soumission = pd.DataFrame(y_pred_soumission, columns=["obs_id", "eqt_code_cat"])
    return(df_y_pred_soumission)

df_y_pred=transfo_format_predi(preds)
df_y_pred.to_csv("/kaggle/working/predictions_TFT.csv", index=False)
